# Figure 1b — charge- and doping-dependent surface recombination in InP

This notebook reproduces the calculated surface-recombination map used in the lower panel of **Figure 1** of the manuscript. The device schematics in Figure 1a are graphical artwork and are not generated here.

The calculation uses the SURPASS v1.0.0 surface electrostatics and energy-resolved surface-SRH model. All plotted quantities are model outputs; the experimental n- and p-InP doping densities are overlaid as reference markers.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import LogFormatterMathtext
from scipy.constants import h, c
from IPython.display import display

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src'/'surpass').exists()), None)
if root is None:
    raise FileNotFoundError('Could not locate the SURPASS project root.')
sys.path.insert(0, str(root/'src'))

import surpass
from surpass import (
    BulkModel, BulkTransportTable, InterfaceChargeModel, InterfaceDefectModel,
    Layer, OpticalStack, SteadyState1DSolver, SurfaceBoundaryTable,
    SurfaceSRHModel, Wafer, get_optical_material, reconstruct_band_profile,
)

print('surpass version:', surpass.__version__)
print('loaded from:', Path(surpass.__file__).resolve())
if tuple(map(int, surpass.__version__.split('.')[:3])) < (0, 9, 0):
    raise ImportError('This notebook requires surpass 1.0.0 or newer.')

try:
    import scienceplots
    plt.style.use(['science', 'notebook', 'no-latex'])
except ImportError:
    plt.style.use('default')

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'font.size': 10,
    'axes.spines.top': True, 'axes.spines.right': True,
})
output_dir = Path.cwd()/'output'
output_dir.mkdir(parents=True, exist_ok=True)

## 1. Experimental inputs and model switches

Absolute PL from n- and p-InP is not compared because different incident
intensities were used. Every simulated PL map is normalized to a separately
calculated **bare InP** reference of the same wafer polarity, using
$D_{it}=10^{12}$ eV$^{-1}$ cm$^{-2}$ and $Q_f=0$. All points in the map use
the 4-nm PO$_x$/13-nm AlO$_x$ optical stack. The plotted ratio therefore
includes the change in pump coupling caused by adding the dielectric stack,
but remains an internal-PL proxy and does not include stack-dependent
luminescence extraction or detector collection.

The nominal electrostatics omits $Q_{it}$ because the charge-neutrality level
and amphoteric-state distribution of PO$_x$/InP are not independently known.
Setting `INCLUDE_INTERFACE_CHARGE=True` activates an explicitly labelled
sensitivity case; it must not be mistaken for a measured interface model.

The energy-independent capture cross sections
$\sigma_n=10^{-14}$ cm$^2$ and $\sigma_p=10^{-13}$ cm$^2$ originate from the
InP(100) surface-state fit of Adamowicz *et al.*, *Vacuum* **63**, 223--227
(2001), DOI: 10.1016/S0042-207X(01)00195-6. That work considered a chemically
polished InP surface and a U-shaped state distribution. The values are used
here only as literature-motivated starting parameters; they are not measured
capture cross sections for PO$_x$/InP or PO$_x$/AlO$_x$/InP.


In [ ]:
wavelength_nm = 514.0
thickness_um = 620.0
thickness_cm = thickness_um*1e-4
wafer_settings = {
    'n-InP': dict(wafer=Wafer('InP', donor_cm3=8.5e18, thickness_um=thickness_um),
                  intensity_w_cm2=1.0, color='tab:blue'),
    'p-InP': dict(wafer=Wafer('InP', acceptor_cm3=5.1e18, thickness_um=thickness_um),
                  intensity_w_cm2=13.0, color='tab:red'),
}

observations = pd.DataFrame([
    {'wafer':'n-InP', 'PL stack/bare':3.071, 'TRPL bare (ns)':2.55,
     'TRPL stack (ns)':3.40, 'PL after -6e12 corona/before':0.70},
    {'wafer':'p-InP', 'PL stack/bare':0.800, 'TRPL bare (ns)':1.88,
     'TRPL stack (ns)':1.79, 'PL after -6e12 corona/before':1.30},
])
display(observations)

# Adamowicz et al., Vacuum 63, 223-227 (2001), DOI:
# 10.1016/S0042-207X(01)00195-6. Their fit to a chemically polished
# InP(100) surface used sigma_n=1e-14 and sigma_p=1e-13 cm^2. These are
# literature-motivated starting values, not fitted POx/InP parameters.
DIT_MAP = 3e10
SIGMA_N = 1e-14
SIGMA_P = 1e-13
VTH_N = 4.13e7
VTH_P = 1.51e7
tau_srh = 5e-9 #np.inf
defects = InterfaceDefectModel(
    DIT_MAP, SIGMA_N, SIGMA_P, VTH_N, VTH_P,
    shape='constant', energy_points=201,
)
print(defects.source)

INCLUDE_INTERFACE_CHARGE = True
interface_charge = (InterfaceChargeModel(0.5, 0.02)
                    if INCLUDE_INTERFACE_CHARGE else None)
ENABLE_BGN = True

# Active photon-reabsorption probability used to reduce the net radiative
# carrier loss. Use 0.0 for the conservative no-recycling baseline and 0.93
# for the high-recycling sensitivity case estimated in notebook 08.
# The intrinsic radiative-emission rate used as the PL proxy remains Bnp.
PHOTON_RECYCLING_PROBABILITY = 0.93
if not 0.0 <= PHOTON_RECYCLING_PROBABILITY < 1.0:
    raise ValueError('PHOTON_RECYCLING_PROBABILITY must satisfy 0 <= p_rec < 1')

def surface_model(wafer, charge_cm2, local_defects=defects):
    return SurfaceSRHModel(
        BulkModel(wafer, enable_bgn=ENABLE_BGN), charge_cm2, local_defects,
        electrostatics='self_consistent', driving_force='excess_product',
        interface_charge=interface_charge,
    )

## 2. Differential low-injection field-effect map — main-text candidate

At exact equilibrium, both $U_s$ and the excess density vanish. The physically
meaningful low-injection quantity is therefore

$$
S_{\mathrm{eff},0}=\lim_{\Delta n_b\rightarrow0}
\frac{U_s(\Delta n_b)}{\Delta n_b}.
$$

The first cell verifies numerical convergence of this ratio. The map then uses
a small perturbation rather than calling $Delta n=0$. This avoids importing a
silicon-specific $J_0$ convention into a PL/lifetime problem.

In [ ]:
low_injection_probes = np.array([1e6, 1e7, 1e8, 1e9, 1e10])
convergence_rows = []
for label, settings in wafer_settings.items():
    for charge in (-4e12, 0.0, 4e12):
        model = surface_model(settings['wafer'], charge)
        values = np.array([model.state(d).effective_s_cm_s
                           for d in low_injection_probes])
        convergence_rows.append({
            'wafer': label, 'Q/q (cm^-2)': charge,
            'S at 1e6': values[0], 'S at 1e8': values[2],
            'S at 1e10': values[-1],
            'relative span 1e6--1e9': np.ptp(values[:4])/np.mean(values[:4]),
        })
display(pd.DataFrame(convergence_rows))

DELTA_LOW = 1e8

In [ ]:
q_map = np.linspace(-1.5e13, 1.5e13, 81)
doping_map = np.logspace(15, 20, 51)
low_maps = {}

for polarity in ('n', 'p'):
    seff = np.empty((doping_map.size, q_map.size))
    balance = np.empty_like(seff)
    surface_degenerate = np.empty_like(seff, dtype=bool)
    bulk_eta = np.empty(doping_map.size)
    for iy, doping in enumerate(doping_map):
        wafer = Wafer('InP', donor_cm3=doping if polarity=='n' else 0.0,
                      acceptor_cm3=doping if polarity=='p' else 0.0,
                      thickness_um=thickness_um)
        bulk = BulkModel(wafer, enable_bgn=ENABLE_BGN).equilibrium()
        bulk_eta[iy] = bulk.eta_n if polarity == 'n' else bulk.eta_p
        for ix, charge in enumerate(q_map):
            state = surface_model(wafer, charge).state(DELTA_LOW)
            seff[iy, ix] = state.effective_s_cm_s
            balance[iy, ix] = np.log10(state.n_surface_cm3/state.p_surface_cm3)
            surface_degenerate[iy, ix] = ((state.eta_n_surface >= 0) or
                                          (state.eta_p_surface >= 0))
    low_maps[polarity] = dict(seff=seff, balance=balance,
                              surface_degenerate=surface_degenerate,
                              bulk_eta=bulk_eta)

# The surface-only ratio is mathematically allowed to become extreme when the
# extrapolated surface excess density tends to zero. Above about 1e8 cm/s the
# experiment is already transport-limited, so clip only the displayed scale.
# The underlying arrays remain unclipped for diagnostics.
vmin, vmax = 1e-1, 1e8
levels = np.geomspace(vmin, vmax, 28)

fig, axes = plt.subplots(2, 1, figsize=(7.6, 7.4), sharex=True,
                         constrained_layout=True)
for ax, polarity, label, actual in zip(
        axes, ('n','p'), ('n-InP','p-InP'), (8.5e18,5.1e18)):
    data = low_maps[polarity]
    cf = ax.contourf(q_map/1e12, doping_map,
                     np.clip(data['seff'], vmin, vmax), levels=levels,
                     norm=LogNorm(vmin=vmin, vmax=vmax), cmap='turbo', extend='both')
    balance_line = ax.contour(q_map/1e12, doping_map, data['balance'],
                              levels=[0], colors='black', linewidths=1.35)
    #ax.clabel(balance_line, fmt={0:r'$n_s=p_s$'}, fontsize=7, inline=True)
    degenerate_overlay = ax.contourf(
        q_map/1e12,
        doping_map,
        data['surface_degenerate'].astype(float),
        levels=[0.5, 1.5],
        colors=[(0.65, 0.65, 0.65, 0.12)],  # transparent grey fill
        hatches=['//'],
    )
    
    degenerate_overlay.set_edgecolor((0.35, 0.35, 0.35, 0.50))
    degenerate_overlay.set_linewidth(0.0)
    # Bulk degeneracy onset eta_majority=0, interpolated in log doping.
    eta = data['bulk_eta']
    idx = np.flatnonzero(np.diff(np.signbit(eta)))
    if idx.size:
        i = idx[0]
        log_ndeg = np.interp(0, eta[i:i+2], np.log10(doping_map[i:i+2]))
        ndeg = 10**log_ndeg
        ax.axhline(ndeg, color='crimson', lw=1.3)
        #ax.annotate('bulk degeneracy', xy=(-11.5, ndeg*1.7), xytext=(-7.5, ndeg/1.7),
        #            color='crimson', fontsize=8,
        #            arrowprops=dict(arrowstyle='->', color='crimson', lw=1.1),
        #            bbox=dict(facecolor='white', alpha=.72, edgecolor='none',
        #                      boxstyle='round,pad=.2'))
    ax.axhline(actual, color='white', lw=1.4, ls='--')
    #ax.text(-14.3, actual*1.12, f'wafer: {actual:.2g} cm$^{{-3}}$', fontsize=7,
     #       bbox=dict(facecolor='white', alpha=.75, edgecolor='none',
      #                boxstyle='round,pad=.2'))
    ax.set_yscale('log')
    ax.set_ylabel(r'Doping density (cm$^{-3}$)')
    ax.set_title(label)
axes[1].invert_yaxis()
axes[1].set_xlabel(r'Total external charge, $Q_{ext}/q$ ($10^{12}$ cm$^{-2}$)')
fig.colorbar(cf, ax=axes, pad=.02, ticks=10.0**np.arange(
    int(np.log10(vmin)), int(np.log10(vmax))+1), format=LogFormatterMathtext(),
    label=rf'$S_{{eff,0}}= U_s/\Delta n_b$ at $\Delta n_b \rightarrow$ 0 cm$^{{-3}}$ (cm s$^{{-1}}$)')
fig.savefig(output_dir/'paper_inp_main_low_injection_map.png', bbox_inches='tight')

### Interpretation

The topology is the robust result. Fixed charge suppresses recombination when
it drives one carrier type away from the surface. Heavy bulk doping shifts the
$n_s=p_s$ contour to larger $|Q_{ext}|$: much more sheet charge is required to
invert a highly doped wafer than a lightly doped wafer. The hatched regions
show where either surface reduced Fermi energy is non-negative, so the
Fermi--Dirac treatment is actually used rather than merely flagged.

The displayed scale is clipped at $10^{-1}$ and $10^8$ cm s$^{-1}$; the
unclipped values remain available in `low_maps`. The upper bound denotes a
regime in which a real wafer would be limited by diffusive carrier delivery.

The map does not establish the initial charge of the experimental stack. A
negative-corona sweep can have the observed local slope even when it starts at
$Q_f=0$.